# NSFnets 案例 2：圆柱绕流（2D 非定常）

**基于物理信息神经网络（PINN）求解不可压 Navier-Stokes 方程**

- 流动类型：层流圆柱绕流尾迹（Re = 100）
- 维度：2D 非定常 (x, y, t)，含时间导数项
- 运动粘度：ν = 0.01
- 网络结构：[3, 50, 50, 50, 50, 3] — 4 个隐藏层，每层 50 个神经元
- 损失函数：α·L_initial（初始条件）+ β·L_boundary（边界条件）+ L_residual（PDE 残差），α=β=100

**数据依赖**：需要 `cylinder_nektar_wake.mat`（Nektar++ CFD 模拟数据）
下载地址：https://github.com/maziarraissi/PINNs/tree/master/main/Data

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import scipy.io
import time

from nsfnet_module import NSFnet2DUnsteady, relative_l2_error

np.random.seed(1234)
print('Imports complete.')

## 1. 加载并预处理 CFD 数据

数据来源为 Nektar++ 在 Re=100 下的圆柱绕流 DNS 模拟。
提取子域：x∈[1,8]、y∈[-2,2]、t∈[0,7]，即圆柱近尾迹区域。

In [ ]:
# Load CFD data — adjust path if needed
import os
data_path = '../Data/cylinder_nektar_wake.mat'
if not os.path.exists(data_path):
    # Alternative paths
    alt_paths = [
        'cylinder_nektar_wake.mat',
        '../../Data/cylinder_nektar_wake.mat',
        '../untitled/PINNs-master/main/Data/cylinder_nektar_wake.mat'
    ]
    for p in alt_paths:
        if os.path.exists(p):
            data_path = p
            break
    else:
        raise FileNotFoundError(
            'cylinder_nektar_wake.mat not found. '
            'Download from: https://github.com/maziarraissi/PINNs/tree/master/main/Data'
        )

data = scipy.io.loadmat(data_path)

U_star = data['U_star']  # N × 2 × T (u, v components)
P_star = data['p_star']  # N × T (pressure)
t_star = data['t']       # T × 1
X_star = data['X_star']  # N × 2 (x, y coordinates)

N = X_star.shape[0]
T = t_star.shape[0]
print(f'Spatial points: {N}, Time steps: {T}')

# Rearrange data into flat arrays
XX = np.tile(X_star[:, 0:1], (1, T))  # N × T
YY = np.tile(X_star[:, 1:2], (1, T))  # N × T
TT = np.tile(t_star, (1, N)).T         # N × T

UU = U_star[:, 0, :]  # N × T (u)
VV = U_star[:, 1, :]  # N × T (v)
PP = P_star            # N × T (p)

# Flatten
x = XX.flatten()[:, None]  # NT × 1
y = YY.flatten()[:, None]  # NT × 1
t = TT.flatten()[:, None]  # NT × 1
u = UU.flatten()[:, None]  # NT × 1
v = VV.flatten()[:, None]  # NT × 1
p = PP.flatten()[:, None]  # NT × 1

# Filter domain: x∈[1,8], y∈[-2,2], t∈[0,7]
data_all = np.concatenate([x, y, t, u, v, p], 1)
data_filtered = data_all[data_all[:, 2] <= 7]
data_filtered = data_filtered[data_filtered[:, 0] >= 1]
data_filtered = data_filtered[data_filtered[:, 0] <= 8]
data_filtered = data_filtered[data_filtered[:, 1] >= -2]
data_domain = data_filtered[data_filtered[:, 1] <= 2]
print(f'Domain data points: {data_domain.shape[0]}')

In [ ]:
# Extract initial condition (t=0)
data_t0 = data_domain[data_domain[:, 2] == 0]

# Extract boundary points (x=1, x=8, y=-2, y=2)
data_y1 = data_domain[data_domain[:, 0] == 1]
data_y8 = data_domain[data_domain[:, 0] == 8]
data_x_lo = data_domain[data_domain[:, 1] == -2]
data_x_hi = data_domain[data_domain[:, 1] == 2]
data_sup_b = np.concatenate([data_y1, data_y8, data_x_lo, data_x_hi], 0)

# Interior training points (random sample)
N_train = 140000
idx = np.random.choice(data_domain.shape[0], N_train, replace=False)

# Prepare training data arrays
def col(arr, i):
    return arr[:, i].reshape(-1, 1).astype(np.float32)

# Initial condition
x0_train = col(data_t0, 0); y0_train = col(data_t0, 1)
t0_train = col(data_t0, 2)
u0_train = col(data_t0, 3); v0_train = col(data_t0, 4)

# Boundary
xb_train = col(data_sup_b, 0); yb_train = col(data_sup_b, 1)
tb_train = col(data_sup_b, 2)
ub_train = col(data_sup_b, 3); vb_train = col(data_sup_b, 4)

# Interior
x_train = col(data_domain[idx], 0)
y_train = col(data_domain[idx], 1)
t_train = col(data_domain[idx], 2)

print(f'Initial points: {x0_train.shape[0]}')
print(f'Boundary points: {xb_train.shape[0]}')
print(f'Interior points: {x_train.shape[0]}')

## 2. 构建模型

网络输入 (x,y,t) → 输出 (u,v,p)

In [ ]:
layers = [3, 50, 50, 50, 50, 3]  # Input: (x, y, t), Output: (u, v, p)

model2d = NSFnet2DUnsteady(
    x0_train, y0_train, t0_train, u0_train, v0_train,
    xb_train, yb_train, tb_train, ub_train, vb_train,
    x_train, y_train, t_train, layers,
    nu=0.01, alpha=100.0, beta=100.0
)

total_params = sum(int(np.prod(p.shape)) for p in model2d.net.trainable_params())
print(f'Trainable parameters: {total_params}')

## 3. 训练

训练策略：Adam 渐进降低学习率 → L-BFGS 二阶精调

In [ ]:
print('=== Phase 1: Adam LR=1e-3 ===')
model2d.adam_train(nIter=5000, learning_rate=1e-3, print_every=500)

In [ ]:
print('=== Phase 2: Adam LR=1e-4 ===')
model2d.adam_train(nIter=5000, learning_rate=1e-4, print_every=500)

In [ ]:
print('=== Phase 3: Adam LR=1e-5 ===')
model2d.adam_train(nIter=50000, learning_rate=1e-5, print_every=5000)

In [ ]:
print('=== Phase 4: Adam LR=1e-6 ===')
model2d.adam_train(nIter=50000, learning_rate=1e-6, print_every=5000)

In [ ]:
print('=== Phase 5: L-BFGS ===')
model2d.lbfgs_train(maxiter=50000)

## 4. 模型评估

在 t = t_star[100] 时刻的快照上对比预测值与 CFD 数据

In [ ]:
# Test: single snapshot at t = t_star[100]
snap = np.array([100])
x_test = X_star[:, 0:1].astype(np.float32)
y_test = X_star[:, 1:2].astype(np.float32)
t_test = TT[:, snap].astype(np.float32)

# True values
u_test = U_star[:, 0, snap].astype(np.float32)[:, None]
v_test = U_star[:, 1, snap].astype(np.float32)[:, None]
p_test = P_star[:, snap].astype(np.float32)[:, None]

# Predict
t0 = time.time()
u_pred, v_pred, p_pred = model2d.predict(x_test, y_test, t_test)
print(f'Prediction time: {time.time() - t0:.2f}s')

In [ ]:
# Relative L2 errors
error_u = relative_l2_error(u_pred, u_test)
error_v = relative_l2_error(v_pred, v_test)
error_p = relative_l2_error(p_pred, p_test)

print(f'Relative L2 Error — u: {error_u:.3e}')
print(f'Relative L2 Error — v: {error_v:.3e}')
print(f'Relative L2 Error — p: {error_p:.3e}')

print('\n=== Expected performance (original TF1 paper) ===')
print('Error u: ~1e-2 — 5e-2')
print('Error v: ~1e-2 — 5e-2')
print('Error p: ~1e-2 — 1e-1')

## 总结

本 Notebook 实现了 2D 非定常圆柱绕流案例：
- **数据驱动**：使用 Nektar++ CFD 数据作为初始条件和边界条件
- **时空耦合**：通过 ∂u/∂t、∂v/∂t 项处理含时流动
- **域筛选**：提取近尾迹区域 [1,8] × [-2,2] × [0,7]
- **14 万训练点**：内部配点用于计算 PDE 残差